# 원하는 포즈로 이미지 만드는 도구 (ControlNet + OpenPose)참조 사진에서 **사람의 관절(포즈)** 만 뽑아내고, 그 포즈를 유지한 채 **전혀 다른 인물·의상·배경**의 이미지를 만드는 도구입니다.**흐름은 네 단계입니다.**1. 참조 이미지 준비 → 2. OpenPose로 관절 추출 → 3. ControlNet 조건으로 이미지 생성 → 4. 결과 저장**시작 전 반드시 할 일**- 상단 메뉴 `런타임` → `런타임 유형 변경` → 하드웨어 가속기를 **GPU (T4 이상)** 로 설정하세요. CPU로는 돌지 않습니다.- 셀은 **위에서 아래로 순서대로** 실행합니다 (`Shift + Enter`).

## 셀 0. 런타임 점검GPU가 잡혔는지, 메모리가 얼마인지 먼저 확인합니다. 이 결과에 따라 아래에서 쓸 모델 등급이 달라집니다.

In [ ]:
# [셀 0] 이 셀이 하는 일: GPU가 연결됐는지 확인하고 VRAM 용량을 출력한다.!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "GPU 없음 → 런타임 유형을 GPU로 바꾸세요"import torchprint("torch:", torch.__version__, "| CUDA 사용 가능:", torch.cuda.is_available())if torch.cuda.is_available():    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3    print(f"VRAM: {vram:.1f} GB")

## 셀 1. 라이브러리 설치`diffusers`(생성 모델), `controlnet_aux`(포즈 추출기)를 설치합니다. 2~3분 걸립니다.설치 후 *"세션을 다시 시작하라"* 는 안내가 뜨면 재시작한 뒤 **셀 2부터** 이어서 실행하면 됩니다.

In [ ]:
# [셀 1] 이 셀이 하는 일: 이미지 생성(diffusers)과 포즈 추출(controlnet_aux)에 필요한 패키지를 설치한다.%pip -q install -U diffusers transformers accelerate safetensors sentencepiece protobuf peft%pip -q install -U controlnet_aux opencv-python-headlessprint("설치 완료")

## 셀 2. 작업 폴더 만들기결과물이 흩어지지 않게 `samples/` 폴더를 만들어 두고, 여기에만 저장합니다.

In [ ]:
# [셀 2] 이 셀이 하는 일: 결과 저장용 폴더를 만들고 공통 설정값을 정한다.import os, torchfrom PIL import ImageWORK = "/content/pose-image-tool"SAMPLES = os.path.join(WORK, "samples")os.makedirs(SAMPLES, exist_ok=True)os.chdir(WORK)WIDTH, HEIGHT = 768, 1024   # 인물은 세로가 길어야 포즈가 안 잘린다SEED = 1234                 # 같은 값이면 같은 결과 → 비교 실험할 때 반드시 고정print("작업 폴더:", WORK)

## 셀 3. 참조 이미지 준비**둘 중 하나**를 고르면 됩니다.- **A. 내 사진 쓰기**: 아래 셀을 실행하면 업로드 창이 뜹니다. 사람 전신이 나온 사진이 가장 잘 됩니다.- **B. 사진 없이 하기**: 업로드 창에서 취소하면, 미리 그려 둔 **합성 스켈레톤**(`samples/pose_01.png`)을 내려받아 씁니다. 이미 OpenPose 형식이라 셀 4의 추출을 건너뛰어도 됩니다.

In [ ]:
# [셀 3] 이 셀이 하는 일: 참조 이미지를 업로드받는다. 업로드를 취소하면 합성 스켈레톤을 대신 내려받는다.from PIL import Imageimport io, urllib.requestref_img = Nonetry:    from google.colab import files    up = files.upload()          # ← 파일 선택 창. 그냥 닫으면 아래 B안으로 넘어간다    if up:        name = list(up.keys())[0]        ref_img = Image.open(io.BytesIO(up[name])).convert("RGB")        print("업로드한 참조 이미지:", name, ref_img.size)except Exception as e:    print("업로드 건너뜀:", e)USE_SYNTHETIC = ref_img is Noneif USE_SYNTHETIC:    # 저장소에 들어 있는 합성 스켈레톤을 쓴다 (본인 깃허브 주소로 바꿔도 된다)    url = "https://raw.githubusercontent.com/daum0809/pose-image-tool/main/samples/pose_01.png"    try:        urllib.request.urlretrieve(url, "samples/pose_01.png")        ref_img = Image.open("samples/pose_01.png").convert("RGB")        print("합성 스켈레톤을 참조로 사용합니다.", ref_img.size)    except Exception as e:        print("내려받기 실패 — samples/pose_01.png 를 직접 업로드하세요.", e)display(ref_img.resize((ref_img.width // 2, ref_img.height // 2)))

## 셀 4. 포즈(관절) 추출사진에서 사람의 관절 위치만 뽑아 **막대인간 그림(포즈 맵)** 으로 바꿉니다. 이 그림이 ControlNet의 조건 입력입니다.얼굴·옷·배경 정보는 여기서 전부 버려지고, **자세만** 남는다는 점이 핵심입니다.

In [ ]:
# [셀 4] 이 셀이 하는 일: OpenPose로 참조 이미지에서 관절을 추출해 포즈 맵(pose_01.png)으로 저장한다.if USE_SYNTHETIC:    pose_map = ref_img.resize((WIDTH, HEIGHT))    print("이미 포즈 맵이므로 추출을 건너뜁니다.")else:    from controlnet_aux import OpenposeDetector    openpose = OpenposeDetector.from_pretrained("lllyasviel/Annotators")    pose_map = openpose(        ref_img,        detect_resolution=512,     # 관절을 찾을 때 쓰는 해상도        image_resolution=max(WIDTH, HEIGHT),        include_body=True,        include_hand=False,        # 손가락까지 쓰려면 True (느려짐)        include_face=False,    ).resize((WIDTH, HEIGHT))pose_map.save("samples/pose_01.png")print("포즈 맵 저장: samples/pose_01.png")display(pose_map.resize((WIDTH // 2, HEIGHT // 2)))

## 셀 5. 모델 등급 선택ControlNet-포즈를 돌리는 방법은 여러 가지인데, **GPU 사정에 따라 되는 조합이 다릅니다.** 그래서 3등급을 준비했습니다.| 등급 | 조합 | 필요 VRAM | 비고 ||---|---|---|---|| `A` | FLUX 계열 + ControlNet Union | 16GB+ | 품질 최상, 모델 ID가 바뀔 수 있어 실패 가능 || `B` | SDXL + openpose ControlNet | 12GB+ | 품질·안정성 균형 (**권장 기본값**) || `C` | SD1.5 + openpose ControlNet | 6GB+ | 가장 가볍고 확실하게 돌아감 |`TIER = "A"` 로 두면 A → B → C 순으로 자동 시도하고, 되는 지점에서 멈춥니다. 시간이 없으면 처음부터 `"B"` 를 권합니다.

In [ ]:
# [셀 5] 이 셀이 하는 일: 사용할 모델 등급과 모델 ID를 정한다. 실행되는 코드는 없고 설정만 한다.TIER = "A"          # "A" | "B" | "C"  (A가 실패하면 자동으로 B, C 순서로 내려감)# 등급 A: FLUX 계열. 모델 카드에서 정확한 저장소 이름을 확인해 앞쪽에 적어 두면 그것부터 시도한다.FLUX_BASE_CANDIDATES = [    "black-forest-labs/FLUX.2-klein",          # 과제에서 지정한 모델 (이름이 다르면 수정)    "black-forest-labs/FLUX.1-schnell",        # 대체 1 (라이선스 개방, 4스텝)    "black-forest-labs/FLUX.1-dev",            # 대체 2 (게이티드 → HF 토큰 필요)]FLUX_CONTROLNET = "Shakker-Labs/FLUX.1-dev-ControlNet-Union-Pro"FLUX_POSE_MODE  = 4        # Union 모델에서 4번이 openpose 채널# 등급 B: SDXLSDXL_BASE       = "stabilityai/stable-diffusion-xl-base-1.0"SDXL_CONTROLNET = "thibaud/controlnet-openpose-sdxl-1.0"SDXL_VAE        = "madebyollin/sdxl-vae-fp16-fix"# 등급 C: SD1.5SD15_BASE       = "stable-diffusion-v1-5/stable-diffusion-v1-5"SD15_CONTROLNET = "lllyasviel/control_v11p_sd15_openpose"print("선택한 등급:", TIER)

### (선택) Hugging Face 토큰게이티드 모델(FLUX.1-dev 등)을 쓸 때만 필요합니다. 토큰은 huggingface.co → Settings → Access Tokens 에서 발급합니다.필요 없으면 이 셀은 건너뛰세요.

In [ ]:
# [선택] 이 셀이 하는 일: 게이티드 모델 접근을 위해 Hugging Face에 로그인한다.# from huggingface_hub import login# login()   # 주석을 풀고 실행하면 토큰 입력창이 뜬다

## 셀 6. 파이프라인 불러오기가장 오래 걸리는 단계입니다(모델 내려받기 5~15분). 한 번 받으면 세션이 살아 있는 동안 다시 받지 않습니다.`enable_model_cpu_offload()` 는 GPU 메모리가 모자랄 때 쓰지 않는 부분을 CPU로 잠시 내려두는 기능입니다. 무료 Colab에서는 거의 필수입니다.

In [ ]:
# [셀 6] 이 셀이 하는 일: 선택한 등급의 ControlNet 파이프라인을 메모리에 올린다. 실패하면 한 등급 아래로 자동 이동한다.import torch, gcfrom diffusers import (ControlNetModel, AutoencoderKL,                       StableDiffusionXLControlNetPipeline,                       StableDiffusionControlNetPipeline)pipe, LOADED_TIER = None, Nonedef _try_flux():    from diffusers import FluxControlNetModel, FluxControlNetPipeline    cn = FluxControlNetModel.from_pretrained(FLUX_CONTROLNET, torch_dtype=torch.bfloat16)    for base in FLUX_BASE_CANDIDATES:        try:            p = FluxControlNetPipeline.from_pretrained(base, controlnet=cn, torch_dtype=torch.bfloat16)            p.enable_model_cpu_offload()            print("FLUX 로드 성공:", base)            return p        except Exception as e:            print(f"  - {base} 실패: {type(e).__name__} {str(e)[:120]}")    raise RuntimeError("FLUX 계열 로드 실패")def _try_sdxl():    cn  = ControlNetModel.from_pretrained(SDXL_CONTROLNET, torch_dtype=torch.float16)    vae = AutoencoderKL.from_pretrained(SDXL_VAE, torch_dtype=torch.float16)    p = StableDiffusionXLControlNetPipeline.from_pretrained(        SDXL_BASE, controlnet=cn, vae=vae, torch_dtype=torch.float16, variant="fp16")    p.enable_model_cpu_offload()    print("SDXL 로드 성공")    return pdef _try_sd15():    cn = ControlNetModel.from_pretrained(SD15_CONTROLNET, torch_dtype=torch.float16)    p = StableDiffusionControlNetPipeline.from_pretrained(        SD15_BASE, controlnet=cn, torch_dtype=torch.float16, safety_checker=None)    p.enable_model_cpu_offload()    print("SD1.5 로드 성공")    return porder = {"A": ["A", "B", "C"], "B": ["B", "C"], "C": ["C"]}[TIER]loaders = {"A": _try_flux, "B": _try_sdxl, "C": _try_sd15}for t in order:    try:        print(f"[{t}] 등급 로드 시도 중...")        pipe, LOADED_TIER = loaders[t](), t        break    except Exception as e:        print(f"[{t}] 실패 → 다음 등급으로: {type(e).__name__} {str(e)[:150]}")        pipe = None        gc.collect(); torch.cuda.empty_cache()assert pipe is not None, "모든 등급 로드 실패 — 런타임을 재시작하고 TIER='C'로 다시 시도하세요."print("최종 사용 등급:", LOADED_TIER)

## 셀 7. 생성 함수등급마다 인자 이름이 조금 다릅니다(FLUX는 `control_image`, SD 계열은 `image`). 그 차이를 함수 하나로 감쌉니다.**조절할 값 두 가지만 기억하면 됩니다.**- `pose_strength` (0~1): 포즈를 얼마나 강하게 따를지. 낮으면 그림은 예쁜데 자세가 어긋나고, 높으면 자세는 맞는데 뻣뻣해집니다. **0.6~0.8 사이가 보통 적당합니다.**- `seed`: 같은 값이면 같은 결과. 프롬프트만 바꿔 비교할 땐 반드시 고정하세요.

In [ ]:
# [셀 7] 이 셀이 하는 일: 등급별 호출 방식 차이를 흡수한 generate() 함수를 정의한다.def generate(prompt, pose_image, negative="", seed=SEED, steps=None, pose_strength=0.7, guidance=None):    g = torch.Generator(device="cuda").manual_seed(seed)    ctrl = pose_image.resize((WIDTH, HEIGHT))    if LOADED_TIER == "A":        out = pipe(prompt=prompt, control_image=ctrl, control_mode=FLUX_POSE_MODE,                   controlnet_conditioning_scale=pose_strength,                   num_inference_steps=steps or 24, guidance_scale=guidance or 3.5,                   width=WIDTH, height=HEIGHT, generator=g)    else:        out = pipe(prompt=prompt, negative_prompt=negative or                   "lowres, bad anatomy, extra limbs, deformed hands, watermark, text",                   image=ctrl, controlnet_conditioning_scale=pose_strength,                   num_inference_steps=steps or 30, guidance_scale=guidance or 7.0,                   width=WIDTH, height=HEIGHT, generator=g)    return out.images[0]print("generate() 준비 완료")

## 셀 8. 첫 생성참조와 **같은 자세**의, **다른 인물**이 나오면 성공입니다.

In [ ]:
# [셀 8] 이 셀이 하는 일: 추출한 포즈로 첫 이미지를 만들고 samples/output_01.png 로 저장한다.prompt_01 = ("a young woman standing in a sunlit library, wearing a beige trench coat, "             "natural light, 35mm photo, sharp focus, full body")img = generate(prompt_01, pose_map, seed=SEED, pose_strength=0.7)img.save("samples/output_01.png")display(img)print("저장: samples/output_01.png")

## 셀 9. 실험 1 — 포즈는 그대로, 프롬프트만 바꾼다같은 뼈대 위에 옷·인물·배경만 갈아 끼웁니다. **자세가 유지되는지**에 주목해서 보세요.

In [ ]:
# [셀 9] 이 셀이 하는 일: 동일한 포즈 맵에 서로 다른 프롬프트 3개를 넣어 결과를 비교한다.variants = {    "a1_astronaut": "an astronaut in a white spacesuit on a red desert planet, cinematic lighting, full body",    "a2_ink":       "traditional korean ink painting of a scholar in hanbok, minimal background, full body",    "a3_3d":        "cute 3d pixar style character, pastel colors, soft studio lighting, full body",}results = {}for name, p in variants.items():    out = generate(p, pose_map, seed=SEED, pose_strength=0.7)   # 시드 고정 = 프롬프트 효과만 본다    out.save(f"samples/{name}.png")    results[name] = out    print(name, "완료")display(*results.values())

## 셀 10. 실험 2 — 프롬프트는 그대로, 포즈만 바꾼다이번엔 두 번째 포즈(`samples/pose_02.png`, 양팔을 든 자세)를 넣습니다.본인 사진을 쓰는 경우엔 셀 3~4를 다른 사진으로 한 번 더 돌려 `pose_02.png` 를 만들면 됩니다.

In [ ]:
# [셀 10] 이 셀이 하는 일: 프롬프트를 고정한 채 포즈 맵만 바꿔 생성하고 output_02.png 로 저장한다.import urllib.requestfrom PIL import Imagetry:    pose_map2 = Image.open("samples/pose_02.png").convert("RGB")except FileNotFoundError:    urllib.request.urlretrieve(        "https://raw.githubusercontent.com/daum0809/pose-image-tool/main/samples/pose_02.png",        "samples/pose_02.png")    pose_map2 = Image.open("samples/pose_02.png").convert("RGB")img2 = generate(prompt_01, pose_map2, seed=SEED, pose_strength=0.7)   # 프롬프트·시드 모두 그대로img2.save("samples/output_02.png")display(pose_map2.resize((WIDTH // 3, HEIGHT // 3)), img2)print("저장: samples/output_02.png")

## 셀 11. (선택) 포즈 강도 비교`pose_strength` 를 0.4 / 0.7 / 1.0 으로 바꿔 세 장을 나란히 봅니다.**어느 값에서 자세가 무너지고, 어느 값에서 그림이 뻣뻣해지는지**가 이 도구의 사용 감각을 잡아 줍니다.

In [ ]:
# [셀 11] 이 셀이 하는 일: 포즈 조건의 세기만 바꿔 가며 생성해 적정값을 눈으로 찾는다.for s in [0.4, 0.7, 1.0]:    o = generate(prompt_01, pose_map, seed=SEED, pose_strength=s)    o.save(f"samples/strength_{s}.png")    print(f"--- pose_strength = {s} ---")    display(o)

## 셀 12. 결과 내려받기`samples/` 폴더를 압축해 내 컴퓨터로 가져옵니다. 이 파일을 풀어서 GitHub 저장소의 `samples/` 에 올리면 제출 준비가 끝납니다.

In [ ]:
# [셀 12] 이 셀이 하는 일: 결과 이미지를 zip으로 묶어 로컬로 내려받는다.!zip -qr samples.zip samplestry:    from google.colab import files    files.download("samples.zip")except Exception as e:    print("Colab이 아니면 수동으로 가져가세요:", e)

## 셀 13. 관찰 기록 — 무엇을 바꿨고 어떻게 달라졌는가> 아래는 **직접 채우는 칸**입니다. 이 노트에서 실제로 본 것만 적으세요.**바꾼 것 1 — 프롬프트 (포즈 고정)**- 넣은 것: `astronaut` / `korean ink painting` / `3d pixar`- 달라진 것:- 자세는 유지됐는가:**바꾼 것 2 — 포즈 (프롬프트 고정)**- 넣은 것: 차렷 자세 → 양팔 든 자세- 달라진 것:- 인물·배경은 유지됐는가:**바꾼 것 3 — pose_strength**- 0.4:- 0.7:- 1.0:**정리**- 결과를 크게 흔든 쪽은 (프롬프트 / 포즈 / 강도) 중 ____ 이었다. 이유는:- 잘 안 따라온 부분(손가락, 겹친 팔다리, 옆모습 등):- 다음에 고친다면: